# TS Missing Folder Distribution

Objective: find the transition-state attempts present in `Data/csvs/First_12000.csv` but absent from `Data/TS/Borane_all.csv`, then locate those discarded identifiers under the known raw/workflow folders.

Done means:
- the attempted/final/missing counts are reproduced;
- every missing identifier has a standardized full ID such as `B_00388_Nu_00001_Cl_00479_0000`;
- files named without conformer IDs, such as `B_00388_Nu_00001_Cl_00479.log`, are also captured via component-level matching;
- matching files are summarized by folder and exported as CSV for failure-reason annotation.

## Setup

The search spaces below implement the known folder patterns:

- `E:/work/B_Cl_Nu/Sum/*`
- `E:/work/B_Cl_Nu/Sum/*/*`
- `E:/work/B_Cl_Nu/Data/first_10000/*`
- `Y:/cold-data/数据备份/B_Cl_Nu/Sum/*`
- `Y:/cold-data/数据备份/B_Cl_Nu/Sum/*/*`
- `Y:/cold-data/数据备份/B_Cl_Nu/Data/*` (filtered to transition-state candidate folders)
- `Y:/cold-data/数据备份/B_Cl_Nu/Data/*/*` (filtered to transition-state candidate folders)

`Ignored/3_Build_DataBase.ipynb` used a fallback for TS logs in `Sum/TS_needIRC`: it first looked for `B_{B}_Nu_{N}_Cl_{Cl}.log`, and only then tried `B_{B}_Nu_{N}_Cl_{Cl}_{conf}.log`. To preserve that provenance behavior, this notebook matches both full-conf filenames and component-only filenames. Component-only hits are flagged because they can map to more than one attempted conformer.

In [1]:
from pathlib import Path
import re
import time
import pandas as pd


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Data" / "csvs" / "First_12000.csv").exists() and (candidate / "Data" / "TS" / "Borane_all.csv").exists():
            return candidate
    raise FileNotFoundError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
ATTEMPTED_CSV = PROJECT_ROOT / "Data" / "csvs" / "First_12000.csv"
FINAL_TS_CSV = PROJECT_ROOT / "Data" / "TS" / "Borane_all.csv"
OUTPUT_DIR = PROJECT_ROOT / "Data" / "csvs"

SEARCH_SPACES = [
    {
        "search_space": "Sum/*",
        "root": Path(r"E:/work/B_Cl_Nu/Sum"),
        "depth": 1,
    },
    {
        "search_space": "Sum/*/*",
        "root": Path(r"E:/work/B_Cl_Nu/Sum"),
        "depth": 2,
    },
    {
        "search_space": "Data/first_10000/*",
        "root": Path(r"E:/work/B_Cl_Nu/Data/first_10000"),
        "depth": 1,
    },
    {
        "search_space": "Y backup Sum/*",
        "root": Path(r"Y:/cold-data/数据备份/B_Cl_Nu/Sum"),
        "depth": 1,
    },
    {
        "search_space": "Y backup Sum/*/*",
        "root": Path(r"Y:/cold-data/数据备份/B_Cl_Nu/Sum"),
        "depth": 2,
    },
    {
        "search_space": "Y backup Data/*",
        "root": Path(r"Y:/cold-data/数据备份/B_Cl_Nu/Data"),
        "depth": 1,
        "include_relative_folder_re": r"(TS|IRC|ENG|fail|dead|uncertain|OtherConf|Combined|React|yqc|Inv)",
    },
    {
        "search_space": "Y backup Data/*/*",
        "root": Path(r"Y:/cold-data/数据备份/B_Cl_Nu/Data"),
        "depth": 2,
        "include_relative_folder_re": r"(TS|IRC|ENG|fail|dead|uncertain|OtherConf|Combined|React|yqc|Inv)",
    },
]

ID_COLS = ["B_Index", "N_Index", "Cl_Index", "B_N_Cl_conf", "Cl_r_conf"]
COMPONENT_COLS = ["B_Index", "N_Index", "Cl_Index"]
FULL_IDENTIFIER_RE = re.compile(r"(B_\d{5}_Nu_\d{5}_Cl_\d{5}_\d{4})(?=\D|$)")
COMPONENT_IDENTIFIER_RE = re.compile(r"(B_\d{5}_Nu_\d{5}_Cl_\d{5})(?=\D|$)")
TS_CANDIDATE_FOLDER_RE = re.compile(r"(TS|IRC|ENG|fail|dead|uncertain|OtherConf|Combined|React)", re.I)

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 100)
print(PROJECT_ROOT)

D:\OneDrive_all\OneDrive\Work\work_part\22.Borane_Radical_Database


## Identifier Construction

`First_12000.csv` and `Borane_all.csv` share the same component and conformer columns.

Full identifier:

`B_{B_Index:05d}_Nu_{N_Index:05d}_Cl_{Cl_Index:05d}_{B_N_Cl_conf:02d}{Cl_r_conf:02d}`

Component identifier:

`B_{B_Index:05d}_Nu_{N_Index:05d}_Cl_{Cl_Index:05d}`

In [2]:
def _as_int(value):
    if pd.isna(value):
        raise ValueError("Identifier column contains NaN")
    return int(float(value))


def add_identifiers(df: pd.DataFrame) -> pd.DataFrame:
    missing = [col for col in ID_COLS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required identifier columns: {missing}")
    out = df.copy()
    out["component_id"] = [
        f"B_{_as_int(b):05d}_Nu_{_as_int(n):05d}_Cl_{_as_int(cl):05d}"
        for b, n, cl in out[COMPONENT_COLS].itertuples(index=False, name=None)
    ]
    out["identifier"] = [
        f"{component}_{_as_int(bconf):02d}{_as_int(clconf):02d}"
        for component, bconf, clconf in zip(out["component_id"], out["B_N_Cl_conf"], out["Cl_r_conf"])
    ]
    return out


attempted = add_identifiers(pd.read_csv(ATTEMPTED_CSV))
final_ts = add_identifiers(pd.read_csv(FINAL_TS_CSV))

attempted_ids = set(attempted["identifier"])
final_ids = set(final_ts["identifier"])
missing_ids = attempted_ids - final_ids
unexpected_final_ids = final_ids - attempted_ids

counts = pd.DataFrame(
    [
        {"metric": "attempted_rows", "value": len(attempted)},
        {"metric": "attempted_unique_identifiers", "value": attempted["identifier"].nunique()},
        {"metric": "attempted_unique_components", "value": attempted["component_id"].nunique()},
        {"metric": "final_ts_rows", "value": len(final_ts)},
        {"metric": "final_ts_unique_identifiers", "value": final_ts["identifier"].nunique()},
        {"metric": "final_ts_unique_components", "value": final_ts["component_id"].nunique()},
        {"metric": "missing_from_final_unique_identifiers", "value": len(missing_ids)},
        {"metric": "final_not_in_attempted_unique_identifiers", "value": len(unexpected_final_ids)},
    ]
)
counts

,metric,value
0,attempted_rows,12000
1,attempted_unique_identifiers,12000
2,attempted_unique_components,12000
3,final_ts_rows,9237
4,final_ts_unique_identifiers,9237
5,final_ts_unique_components,9237
6,missing_from_final_unique_identifiers,2763
7,final_not_in_attempted_unique_identifiers,0


In [3]:
if len(attempted) != attempted["identifier"].nunique():
    dupes = attempted[attempted["identifier"].duplicated(keep=False)].sort_values("identifier")
    display(dupes.head(20))
    raise ValueError("Attempted table has duplicate full identifiers; inspect duplicates before continuing.")

missing_attempts = attempted[attempted["identifier"].isin(missing_ids)].copy()
missing_attempts = missing_attempts.sort_values(["B_Index", "N_Index", "Cl_Index", "B_N_Cl_conf", "Cl_r_conf"])

missing_component_counts = (
    missing_attempts.groupby("component_id")["identifier"]
    .nunique()
    .rename("missing_identifiers_for_component")
    .reset_index()
)
missing_by_identifier = missing_attempts.set_index("identifier")
missing_by_component = {
    component_id: group["identifier"].tolist()
    for component_id, group in missing_attempts.groupby("component_id", sort=False)
}

print(f"Missing identifiers: {len(missing_attempts)} / {len(attempted)} ({len(missing_attempts) / len(attempted):.2%})")
print(f"Missing components: {missing_attempts['component_id'].nunique()}")
missing_attempts.head()

Missing identifiers: 2763 / 12000 (23.03%)
Missing components: 2763


,B_smiles,B_Index,B_Atomid,N_smiles,N_Index,N_Atomid,Cl_smiles,Cl_Index,Cl_Atomid,B_N_Cl_conf,Cl_r_conf,deltaG_react,component_id,identifier
9237,B,388,0,CCN(CC)CC,0,2,ClCCl,462,0,1,0,-36.720610,B_00388_Nu_00000_Cl_00462,B_00388_Nu_00000_Cl_00462_0100
9806,B,388,0,CCN(CC)CC,0,2,ClC(Cl)c1ncncn1,463,0,1,0,-50.280634,B_00388_Nu_00000_Cl_00463,B_00388_Nu_00000_Cl_00463_0100
9238,B,388,0,CN(C)C,1,1,CC(=O)C(Cl)(Cl)Cl,443,4,0,0,-59.788200,B_00388_Nu_00001_Cl_00443,B_00388_Nu_00001_Cl_00443_0000
9239,B,388,0,CN(C)C,1,1,O=C(Nc1ccccc1)C(Cl)(Cl)Cl,444,10,0,0,-58.340746,B_00388_Nu_00001_Cl_00444,B_00388_Nu_00001_Cl_00444_0000
9240,B,388,0,CN(C)C,1,1,O=C=NC(=O)C(Cl)(Cl)Cl,445,6,0,0,-61.205221,B_00388_Nu_00001_Cl_00445,B_00388_Nu_00001_Cl_00445_0000


## Scan Raw Folders

This cell indexes files whose names contain either a full identifier or a component-only identifier.

Matching rules:

- `full_conf`: the filename contains the exact full identifier, including conformer digits.
- `component_only`: the filename contains only `B_..._Nu_..._Cl_...`; the hit is assigned to every missing attempted conformer with that component and flagged as potentially ambiguous.

For reviewer-facing TS triage, `ts_candidate_hit` is stricter than `has_file_hit`: exact full-conf hits count everywhere, but component-only hits only count as TS-candidate evidence in TS/IRC/ENG/fail/dead/uncertain/Combined/OtherConf/React-style folders. Component-only files in broad structure folders such as `Mols` are retained in the raw hit table but do not inflate the TS-candidate coverage.

In [4]:
def iter_files_for_space(space: dict):
    root = space["root"]
    depth = space["depth"]
    include_relative_folder_re = space.get("include_relative_folder_re")
    include_re = re.compile(include_relative_folder_re, re.I) if include_relative_folder_re else None
    if not root.exists():
        print(f"Missing search root: {root}")
        return

    if depth == 1:
        for folder in sorted(root.iterdir()):
            if not folder.is_dir():
                continue
            relative_folder = folder.relative_to(root).as_posix()
            if include_re and not include_re.search(relative_folder):
                continue
            for file_path in folder.iterdir():
                if file_path.is_file():
                    yield space["search_space"], root, folder, file_path
    elif depth == 2:
        for folder in sorted(root.iterdir()):
            if not folder.is_dir():
                continue
            for subfolder in sorted(folder.iterdir()):
                if not subfolder.is_dir():
                    continue
                relative_folder = subfolder.relative_to(root).as_posix()
                if include_re and not include_re.search(relative_folder):
                    continue
                for file_path in subfolder.iterdir():
                    if file_path.is_file():
                        yield space["search_space"], root, subfolder, file_path
    else:
        raise ValueError(f"Unsupported depth: {depth}")


def extract_file_identifier(name: str):
    full_match = FULL_IDENTIFIER_RE.search(name)
    if full_match:
        full_identifier = full_match.group(1)
        return {
            "file_identifier": full_identifier,
            "file_component_id": full_identifier.rsplit("_", 1)[0],
            "file_id_scope": "full_conf",
        }
    component_match = COMPONENT_IDENTIFIER_RE.search(name)
    if component_match:
        component_id = component_match.group(1)
        return {
            "file_identifier": component_id,
            "file_component_id": component_id,
            "file_id_scope": "component_only",
        }
    return None


start = time.time()
rows = []
scan_stats = []

for space in SEARCH_SPACES:
    scanned_files = 0
    identifier_like_files = 0
    matched_files = 0
    full_identifier_like_files = 0
    component_identifier_like_files = 0
    for search_space, root, folder, file_path in iter_files_for_space(space):
        scanned_files += 1
        parsed = extract_file_identifier(file_path.name)
        if parsed is None:
            continue
        identifier_like_files += 1
        if parsed["file_id_scope"] == "full_conf":
            full_identifier_like_files += 1
            target_identifiers = [parsed["file_identifier"]] if parsed["file_identifier"] in missing_ids else []
        else:
            component_identifier_like_files += 1
            target_identifiers = missing_by_component.get(parsed["file_component_id"], [])
        if not target_identifiers:
            continue
        matched_files += 1
        component_missing_count = len(missing_by_component.get(parsed["file_component_id"], []))
        is_ts_candidate_folder = bool(TS_CANDIDATE_FOLDER_RE.search(folder.relative_to(root).as_posix()))
        is_ts_candidate_hit = parsed["file_id_scope"] == "full_conf" or is_ts_candidate_folder
        for identifier in target_identifiers:
            rows.append(
                {
                    "identifier": identifier,
                    "component_id": parsed["file_component_id"],
                    "match_scope": parsed["file_id_scope"],
                    "is_ts_candidate_folder": is_ts_candidate_folder,
                    "is_ts_candidate_hit": is_ts_candidate_hit,
                    "component_missing_candidates": component_missing_count,
                    "component_match_is_ambiguous": parsed["file_id_scope"] == "component_only" and component_missing_count > 1,
                    "file_identifier": parsed["file_identifier"],
                    "search_space": search_space,
                    "folder": folder.name,
                    "relative_folder": folder.relative_to(root).as_posix(),
                    "file_name": file_path.name,
                    "suffix": file_path.suffix.lower(),
                    "path": str(file_path),
                }
            )
    scan_stats.append(
        {
            "search_space": space["search_space"],
            "root": str(space["root"]),
            "depth": space["depth"],
            "scanned_files": scanned_files,
            "identifier_like_files": identifier_like_files,
            "full_identifier_like_files": full_identifier_like_files,
            "component_identifier_like_files": component_identifier_like_files,
            "matched_files": matched_files,
        }
    )

scan_stats = pd.DataFrame(scan_stats)
file_hits = pd.DataFrame(rows)
print(f"Scan finished in {time.time() - start:.1f} s")
scan_stats

Scan finished in 197.0 s


,search_space,root,depth,scanned_files,identifier_like_files,full_identifier_like_files,component_identifier_like_files,matched_files
0,Sum/*,E:\work\B_Cl_Nu\Sum,1,96208,96208,85552,10656,918
1,Sum/*/*,E:\work\B_Cl_Nu\Sum,2,19723,19723,19549,174,137
2,Data/first_10000/*,E:\work\B_Cl_Nu\Data\first_10000,1,151781,151680,141680,10000,9967
3,Y backup Sum/*,Y:\cold-data\数据备份\B_Cl_Nu\Sum,1,99975,99975,98964,1011,910
4,Y backup Sum/*/*,Y:\cold-data\数据备份\B_Cl_Nu\Sum,2,2725,2525,2507,18,134
5,Y backup Data/*,Y:\cold-data\数据备份\B_Cl_Nu\Data,1,25,3,2,1,0
6,Y backup Data/*/*,Y:\cold-data\数据备份\B_Cl_Nu\Data,2,138730,138376,122094,16282,3830


In [5]:
if file_hits.empty:
    print("No files matched the missing identifiers in the configured search spaces.")
else:
    print(f"Matched file-hit rows: {len(file_hits):,}")
    print(f"Unique matched files: {file_hits['path'].nunique():,}")
    print(f"Matched missing identifiers: {file_hits['identifier'].nunique():,} / {len(missing_attempts):,}")
    display(file_hits["match_scope"].value_counts().rename_axis("match_scope").reset_index(name="file_hit_rows"))
    display(file_hits.head())

Matched file-hit rows: 15,896
Unique matched files: 15,896
Matched missing identifiers: 2,703 / 2,763


,match_scope,file_hit_rows
0,full_conf,13155
1,component_only,2741


,identifier,component_id,match_scope,is_ts_candidate_folder,is_ts_candidate_hit,component_missing_candidates,component_match_is_ambiguous,file_identifier,search_space,folder,relative_folder,file_name,suffix,path
0,B_00388_Nu_00141_Cl_00504_0000,B_00388_Nu_00141_Cl_00504,full_conf,True,True,1,False,B_00388_Nu_00141_Cl_00504_0000,Sum/*,IRC_full,IRC_full,B_00388_Nu_00141_Cl_00504_0000forward.log,.log,E:\work\B_Cl_Nu\Sum\IRC_full\B_00388_Nu_00141_Cl_00504_0000forward.log
1,B_00388_Nu_00141_Cl_00504_0000,B_00388_Nu_00141_Cl_00504,full_conf,True,True,1,False,B_00388_Nu_00141_Cl_00504_0000,Sum/*,IRC_full,IRC_full,B_00388_Nu_00141_Cl_00504_0000reverse.log,.log,E:\work\B_Cl_Nu\Sum\IRC_full\B_00388_Nu_00141_Cl_00504_0000reverse.log
2,B_00388_Nu_00143_Cl_00564_0000,B_00388_Nu_00143_Cl_00564,full_conf,True,True,1,False,B_00388_Nu_00143_Cl_00564_0000,Sum/*,IRC_full,IRC_full,B_00388_Nu_00143_Cl_00564_0000forward.log,.log,E:\work\B_Cl_Nu\Sum\IRC_full\B_00388_Nu_00143_Cl_00564_0000forward.log
3,B_00388_Nu_00143_Cl_00564_0000,B_00388_Nu_00143_Cl_00564,full_conf,True,True,1,False,B_00388_Nu_00143_Cl_00564_0000,Sum/*,IRC_full,IRC_full,B_00388_Nu_00143_Cl_00564_0000reverse.log,.log,E:\work\B_Cl_Nu\Sum\IRC_full\B_00388_Nu_00143_Cl_00564_0000reverse.log
4,B_00388_Nu_00184_Cl_00499_0000,B_00388_Nu_00184_Cl_00499,full_conf,True,True,1,False,B_00388_Nu_00184_Cl_00499_0000,Sum/*,IRC_full,IRC_full,B_00388_Nu_00184_Cl_00499_0000forward.gjf,.gjf,E:\work\B_Cl_Nu\Sum\IRC_full\B_00388_Nu_00184_Cl_00499_0000forward.gjf


## Folder Distribution

`matched_identifiers` is the all-artifact distribution number. `matched_files` counts unique file paths, because component-only files can map to several missing conformers.

For TS failure triage, prefer `ts_candidate_identifiers`. It includes exact full-conf hits plus component-only hits in TS/IRC-like folders. `component_non_ts_artifact_identifiers` keeps broad structure-folder hits visible without mixing them into TS-candidate coverage.

In [6]:
def _nunique_where(df, value_col, mask_col, mask_value):
    return df.loc[df[mask_col] == mask_value, value_col].nunique()


if file_hits.empty:
    folder_distribution = pd.DataFrame(
        columns=[
            "search_space",
            "relative_folder",
            "matched_identifiers",
            "full_conf_identifiers",
            "component_inferred_identifiers",
            "ambiguous_component_identifiers",
            "matched_components",
            "matched_files",
            "example_identifiers",
        ]
    )
else:
    folder_rows = []
    for (search_space, relative_folder), group in file_hits.groupby(["search_space", "relative_folder"], dropna=False):
        folder_rows.append(
            {
                "search_space": search_space,
                "relative_folder": relative_folder,
                "matched_identifiers": group["identifier"].nunique(),
                "ts_candidate_identifiers": group.loc[group["is_ts_candidate_hit"], "identifier"].nunique(),
                "full_conf_identifiers": _nunique_where(group, "identifier", "match_scope", "full_conf"),
                "component_inferred_identifiers": _nunique_where(group, "identifier", "match_scope", "component_only"),
                "component_ts_fallback_identifiers": group.loc[
                    (group["match_scope"] == "component_only") & (group["is_ts_candidate_hit"]), "identifier"
                ].nunique(),
                "component_non_ts_artifact_identifiers": group.loc[
                    (group["match_scope"] == "component_only") & (~group["is_ts_candidate_hit"]), "identifier"
                ].nunique(),
                "ambiguous_component_identifiers": group.loc[group["component_match_is_ambiguous"], "identifier"].nunique(),
                "matched_components": group["component_id"].nunique(),
                "matched_files": group["path"].nunique(),
                "example_identifiers": ", ".join(sorted(set(group["identifier"]))[:5]),
            }
        )
    folder_distribution = (
        pd.DataFrame(folder_rows)
        .sort_values(["ts_candidate_identifiers", "matched_identifiers", "matched_files"], ascending=False)
        .reset_index(drop=True)
    )

folder_distribution

,search_space,relative_folder,matched_identifiers,ts_candidate_identifiers,full_conf_identifiers,component_inferred_identifiers,component_ts_fallback_identifiers,component_non_ts_artifact_identifiers,ambiguous_component_identifiers,matched_components,matched_files,example_identifiers
0,Data/first_10000/*,OM,2143,2143,2143,0,0,0,0,2143,4286,"B_00388_Nu_00001_Cl_00443_0000, B_00388_Nu_00001_Cl_00444_0000, B_00388_Nu_00001_Cl_00445_0000, B_00388_Nu_00001_Cl_00446_0000, B_00388_Nu_00001_Cl_00447_0000"
1,Data/first_10000/*,TS,2142,2142,2142,0,0,0,0,2142,2146,"B_00388_Nu_00001_Cl_00443_0000, B_00388_Nu_00001_Cl_00444_0000, B_00388_Nu_00001_Cl_00445_0000, B_00388_Nu_00001_Cl_00446_0000, B_00388_Nu_00001_Cl_00447_0000"
2,Y backup Data/*/*,first_10000/TS,2142,2142,2142,0,0,0,0,2142,2146,"B_00388_Nu_00001_Cl_00443_0000, B_00388_Nu_00001_Cl_00444_0000, B_00388_Nu_00001_Cl_00445_0000, B_00388_Nu_00001_Cl_00446_0000, B_00388_Nu_00001_Cl_00447_0000"
3,Sum/*,TS_ENG,208,208,202,17,17,0,0,208,222,"B_00388_Nu_00141_Cl_00504_0000, B_00388_Nu_00143_Cl_00475_0000, B_00388_Nu_00143_Cl_00487_0000, B_00388_Nu_00143_Cl_00554_0000, B_00388_Nu_00143_Cl_00564_0000"
4,Y backup Sum/*,TS_ENG,208,208,202,17,17,0,0,208,222,"B_00388_Nu_00141_Cl_00504_0000, B_00388_Nu_00143_Cl_00475_0000, B_00388_Nu_00143_Cl_00487_0000, B_00388_Nu_00143_Cl_00554_0000, B_00388_Nu_00143_Cl_00564_0000"
...,...,...,...,...,...,...,...,...,...,...,...,...
170,Y backup Sum/*/*,IRC_full3_/subdir_91,1,1,1,0,0,0,0,1,1,B_00439_Nu_00057_Cl_00501_0000
171,Y backup Sum/*/*,IRC_full3_/subdir_92,1,1,1,0,0,0,0,1,1,B_00439_Nu_00057_Cl_00501_0000
172,Y backup Sum/*/*,IRC_full3_/subdir_96,1,1,1,0,0,0,0,1,1,B_00421_Nu_00220_Cl_00514_0000
173,Y backup Sum/*/*,IRC_full3_/subdir_97,1,1,1,0,0,0,0,1,1,B_00388_Nu_00184_Cl_00563_0000


In [7]:
if file_hits.empty:
    identifier_hits = pd.DataFrame(columns=["identifier"])
else:
    identifier_rows = []
    for identifier, group in file_hits.groupby("identifier", sort=False):
        identifier_rows.append(
            {
                "identifier": identifier,
                "n_hit_files": group["path"].nunique(),
                "n_ts_candidate_hit_files": group.loc[group["is_ts_candidate_hit"], "path"].nunique(),
                "n_hit_full_conf_files": group.loc[group["match_scope"] == "full_conf", "path"].nunique(),
                "n_hit_component_only_files": group.loc[group["match_scope"] == "component_only", "path"].nunique(),
                "n_hit_component_only_ts_fallback_files": group.loc[
                    (group["match_scope"] == "component_only") & (group["is_ts_candidate_hit"]), "path"
                ].nunique(),
                "n_hit_component_only_non_ts_artifact_files": group.loc[
                    (group["match_scope"] == "component_only") & (~group["is_ts_candidate_hit"]), "path"
                ].nunique(),
                "n_hit_folders": group["relative_folder"].nunique(),
                "has_ambiguous_component_hit": bool(group["component_match_is_ambiguous"].any()),
                "hit_match_scopes": " | ".join(sorted(set(group["match_scope"]))),
                "hit_search_spaces": " | ".join(sorted(set(group["search_space"]))),
                "hit_folders": " | ".join(sorted(set(group["relative_folder"]))),
                "hit_suffixes": " | ".join(sorted(set(group["suffix"]))),
                "hit_paths": " | ".join(sorted(set(group["path"]))),
            }
        )
    identifier_hits = pd.DataFrame(identifier_rows)

discarded_identifiers = missing_attempts.merge(missing_component_counts, on="component_id", how="left")
discarded_identifiers = discarded_identifiers.merge(identifier_hits, on="identifier", how="left")

numeric_hit_cols = [
    "n_hit_files",
    "n_ts_candidate_hit_files",
    "n_hit_full_conf_files",
    "n_hit_component_only_files",
    "n_hit_component_only_ts_fallback_files",
    "n_hit_component_only_non_ts_artifact_files",
    "n_hit_folders",
]
for col in numeric_hit_cols:
    discarded_identifiers[col] = discarded_identifiers[col].fillna(0).astype(int)

discarded_identifiers["has_file_hit"] = discarded_identifiers["n_hit_files"] > 0
discarded_identifiers["has_ts_candidate_file_hit"] = discarded_identifiers["n_ts_candidate_hit_files"] > 0
discarded_identifiers["has_full_conf_file_hit"] = discarded_identifiers["n_hit_full_conf_files"] > 0
discarded_identifiers["has_component_only_file_hit"] = discarded_identifiers["n_hit_component_only_files"] > 0
discarded_identifiers["has_component_only_ts_fallback_hit"] = discarded_identifiers["n_hit_component_only_ts_fallback_files"] > 0
discarded_identifiers["has_component_only_non_ts_artifact_hit"] = discarded_identifiers["n_hit_component_only_non_ts_artifact_files"] > 0
discarded_identifiers["has_ambiguous_component_hit"] = discarded_identifiers["has_ambiguous_component_hit"].fillna(False).astype(bool)

for col in ["hit_match_scopes", "hit_search_spaces", "hit_folders", "hit_suffixes", "hit_paths"]:
    discarded_identifiers[col] = discarded_identifiers[col].fillna("")

attempted_status = attempted.merge(
    final_ts[["identifier"]].assign(in_final_ts=True),
    on="identifier",
    how="left",
)
attempted_status["in_final_ts"] = attempted_status["in_final_ts"].fillna(False)
attempted_status = attempted_status.merge(identifier_hits, on="identifier", how="left")

for col in numeric_hit_cols:
    attempted_status[col] = attempted_status[col].fillna(0).astype(int)
attempted_status["has_file_hit"] = attempted_status["n_hit_files"] > 0
attempted_status["has_ts_candidate_file_hit"] = attempted_status["n_ts_candidate_hit_files"] > 0
attempted_status["has_full_conf_file_hit"] = attempted_status["n_hit_full_conf_files"] > 0
attempted_status["has_component_only_file_hit"] = attempted_status["n_hit_component_only_files"] > 0
attempted_status["has_component_only_ts_fallback_hit"] = attempted_status["n_hit_component_only_ts_fallback_files"] > 0
attempted_status["has_component_only_non_ts_artifact_hit"] = attempted_status["n_hit_component_only_non_ts_artifact_files"] > 0
attempted_status["has_ambiguous_component_hit"] = attempted_status["has_ambiguous_component_hit"].fillna(False).astype(bool)

for col in ["hit_match_scopes", "hit_search_spaces", "hit_folders", "hit_suffixes", "hit_paths"]:
    attempted_status[col] = attempted_status[col].fillna("")

coverage = pd.DataFrame(
    [
        {"metric": "discarded_identifiers", "value": len(discarded_identifiers)},
        {"metric": "discarded_with_any_artifact_file_hit", "value": int(discarded_identifiers["has_file_hit"].sum())},
        {"metric": "discarded_with_ts_candidate_file_hit", "value": int(discarded_identifiers["has_ts_candidate_file_hit"].sum())},
        {"metric": "discarded_with_full_conf_file_hit", "value": int(discarded_identifiers["has_full_conf_file_hit"].sum())},
        {"metric": "discarded_with_component_only_file_hit", "value": int(discarded_identifiers["has_component_only_file_hit"].sum())},
        {"metric": "discarded_with_component_only_ts_fallback_hit", "value": int(discarded_identifiers["has_component_only_ts_fallback_hit"].sum())},
        {"metric": "discarded_with_component_only_non_ts_artifact_hit", "value": int(discarded_identifiers["has_component_only_non_ts_artifact_hit"].sum())},
        {"metric": "discarded_with_ambiguous_component_hit", "value": int(discarded_identifiers["has_ambiguous_component_hit"].sum())},
        {"metric": "discarded_without_any_artifact_file_hit", "value": int((~discarded_identifiers["has_file_hit"]).sum())},
        {"metric": "discarded_without_ts_candidate_file_hit", "value": int((~discarded_identifiers["has_ts_candidate_file_hit"]).sum())},
        {"metric": "discarded_any_artifact_file_hit_coverage", "value": discarded_identifiers["has_file_hit"].mean()},
        {"metric": "discarded_ts_candidate_file_hit_coverage", "value": discarded_identifiers["has_ts_candidate_file_hit"].mean()},
        {"metric": "discarded_full_conf_file_hit_coverage", "value": discarded_identifiers["has_full_conf_file_hit"].mean()},
    ]
)
coverage

C:\Users\Jackie\AppData\Local\Temp\ipykernel_66644\1015124593.py:51: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  discarded_identifiers["has_ambiguous_component_hit"] = discarded_identifiers["has_ambiguous_component_hit"].fillna(False).astype(bool)
C:\Users\Jackie\AppData\Local\Temp\ipykernel_66644\1015124593.py:61: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  attempted_status["in_final_ts"] = attempted_status["in_final_ts"].fillna(False)
C:\Users\Jackie\AppData\Local\Temp\ipykernel_66644\1015124593.py:72: FutureWarning: Downcasting object dtype arrays

,metric,value
0,discarded_identifiers,2763.000000
1,discarded_with_any_artifact_file_hit,2703.000000
2,discarded_with_ts_candidate_file_hit,2363.000000
3,discarded_with_full_conf_file_hit,2351.000000
4,discarded_with_component_only_file_hit,2529.000000
5,discarded_with_component_only_ts_fallback_hit,49.000000
6,discarded_with_component_only_non_ts_artifact_hit,2483.000000
7,discarded_with_ambiguous_component_hit,0.000000
8,discarded_without_any_artifact_file_hit,60.000000
9,discarded_without_ts_candidate_file_hit,400.000000


In [8]:
unlocated = discarded_identifiers.loc[
    ~discarded_identifiers["has_ts_candidate_file_hit"],
    ["identifier", "component_id", "B_Index", "N_Index", "Cl_Index", "B_N_Cl_conf", "Cl_r_conf", "deltaG_react"],
]
print(f"Discarded identifiers without TS-candidate file hits: {len(unlocated):,}")
unlocated.head(20)

Discarded identifiers without TS-candidate file hits: 400


,identifier,component_id,B_Index,N_Index,Cl_Index,B_N_Cl_conf,Cl_r_conf,deltaG_react
0,B_00388_Nu_00000_Cl_00462_0100,B_00388_Nu_00000_Cl_00462,388,0,462,1,0,-36.720610
1,B_00388_Nu_00000_Cl_00463_0100,B_00388_Nu_00000_Cl_00463,388,0,463,1,0,-50.280634
173,B_00388_Nu_00019_Cl_00479_0100,B_00388_Nu_00019_Cl_00479,388,19,479,1,0,-29.524753
174,B_00388_Nu_00019_Cl_00510_0100,B_00388_Nu_00019_Cl_00510,388,19,510,1,0,-63.791838
183,B_00388_Nu_00025_Cl_00462_0100,B_00388_Nu_00025_Cl_00462,388,25,462,1,0,-39.197289
184,B_00388_Nu_00025_Cl_00527_0100,B_00388_Nu_00025_Cl_00527,388,25,527,1,0,-50.855800
191,B_00388_Nu_00028_Cl_00462_0100,B_00388_Nu_00028_Cl_00462,388,28,462,1,0,-40.059161
222,B_00388_Nu_00051_Cl_00472_0101,B_00388_Nu_00051_Cl_00472,388,51,472,1,1,-55.517184
223,B_00388_Nu_00051_Cl_00473_0100,B_00388_Nu_00051_Cl_00473,388,51,473,1,0,-58.740902
224,B_00388_Nu_00051_Cl_00493_0100,B_00388_Nu_00051_Cl_00493,388,51,493,1,0,-33.510320


## Export Tables

These files are intentionally review-oriented:

- `attempted_ts_status_from_file_locations.csv`: all 12000 attempted rows with final/missing and file-hit status.
- `discarded_ts_identifiers_file_locations.csv`: only discarded identifiers, ready for manual or scripted failure-reason annotation.
- `discarded_ts_folder_distribution.csv`: folder-level distribution with full-conf, TS-candidate component fallback, and non-TS component artifacts separated.
- `discarded_ts_file_hits.csv`: one row per missing identifier per matched raw/workflow file.

In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

attempted_status_path = OUTPUT_DIR / "attempted_ts_status_from_file_locations.csv"
discarded_path = OUTPUT_DIR / "discarded_ts_identifiers_file_locations.csv"
folder_distribution_path = OUTPUT_DIR / "discarded_ts_folder_distribution.csv"
file_hits_path = OUTPUT_DIR / "discarded_ts_file_hits.csv"
scan_stats_path = OUTPUT_DIR / "discarded_ts_scan_stats.csv"

attempted_status.to_csv(attempted_status_path, index=False)
discarded_identifiers.to_csv(discarded_path, index=False)
folder_distribution.to_csv(folder_distribution_path, index=False)
file_hits.to_csv(file_hits_path, index=False)
scan_stats.to_csv(scan_stats_path, index=False)

for path in [attempted_status_path, discarded_path, folder_distribution_path, file_hits_path, scan_stats_path]:
    print(path.relative_to(PROJECT_ROOT))

Data\csvs\attempted_ts_status_from_file_locations.csv
Data\csvs\discarded_ts_identifiers_file_locations.csv
Data\csvs\discarded_ts_folder_distribution.csv
Data\csvs\discarded_ts_file_hits.csv
Data\csvs\discarded_ts_scan_stats.csv


## First-Pass Interpretation Notes

Use the folder distribution as a triage map rather than a final failure-class table:

- folders such as `TS_fail`, `ts_dead`, and `ts_irc_fail_new` are likely direct failure buckets;
- folders such as `TS_needIRC`, `TS_ENG`, `IRC_full`, and `OtherIRCs` may require log/IRC parsing before assigning a final reason;
- `component_only` hits follow the naming fallback seen in `Ignored/3_Build_DataBase.ipynb`, but they are not conformer-specific and should be treated as inferred evidence when multiple missing conformers share the same component ID.
- `Mols` and similar broad structure folders are preserved as all-artifact hits, but they are not counted as TS-candidate component fallback hits.